In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Italy Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'IT'
NUTS2 = 'Veneto'

In [4]:
YEAR = 2023
MONTH = 'July'
PERIOD = '1st'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,population,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,11.79324,45.35865,2023-07-01,veneto,abano terme,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,"19,349",46.86670,0.329492,0.117878,-0.292611,-0.117878,0.309622,0.067113,-0.289598,-0.067113,0.064020,0.035178,0.049813,0.035178,30.620,36.950,24.29,7.626000,0.600000,8.080000,-0.791538,18.365714,6.326364,21.500769,6.950000,47.303076,51.273032,859.060975,8982.126981,2074.539971,3,170.409233,10.722186,179.966181,0.0,1.063084,31,98.0,36,98.0,30,98.0,12,12,1,6,7,2,0,207,0,0
1,12.04199,45.06525,2023-07-01,veneto,adria,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,"20,233",46.64640,0.460924,0.216659,-0.429246,-0.216659,0.374024,0.132098,-0.364986,-0.132098,0.105025,0.097319,0.075258,0.097319,25.668,33.406,17.93,6.377211,0.808571,8.476395,-1.590923,16.728553,4.271543,21.836459,5.637644,116.060689,116.328961,1336.011096,6701.380530,1644.125334,2,145.603087,-1.354496,180.070371,0.0,1.206604,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,275,0,0
2,10.77673,45.55680,2023-07-01,veneto,affi,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,"2,297",46.81410,0.324080,0.049313,-0.310440,-0.049313,0.424335,0.112184,-0.378497,-0.112184,0.091998,0.053705,0.066999,0.053705,26.740,30.790,22.69,5.821667,0.738000,8.631538,0.720667,17.207647,3.983750,18.918421,6.878889,9.603157,12.076969,518.306948,5306.025387,738.794180,6,207.641691,201.821006,178.733735,0.0,1.815834,31,84.0,30,84.0,30,84.0,10,10,1,6,6,2,0,59,0,0
3,11.96539,45.17531,2023-07-01,veneto,agna,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,"3,400",46.73306,0.335307,0.001666,-0.351311,-0.001666,0.415759,0.089003,-0.405893,-0.089003,0.086590,0.072357,0.063562,0.072357,28.220,34.330,22.11,4.870000,0.610000,7.752222,-2.034615,15.791176,4.124000,21.578571,5.119231,107.449202,107.469327,1289.586229,16631.850238,1165.197945,3,121.193871,0.122498,180.188821,0.0,3.769479,31,99.0,36,99.0,30,99.0,12,12,1,6,7,2,0,151,0,0
4,12.04755,46.30297,2023-07-01,veneto,agordo,1,7,26,2023,0.201299,0.97953,-0.5,-0.866025,0.059241,-0.998244,"4,249",47.84463,0.717695,0.310312,-0.609718,-0.310312,0.665383,0.304986,-0.559188,-0.304986,0.020794,0.021090,0.020415,0.021090,17.000,21.730,12.27,1.921429,-4.077692,5.623333,-2.298000,9.285385,-0.998333,11.544545,0.444545,10.098235,45.995950,957.602493,15023.566702,684.464446,18,124.001040,1470.386475,152.701532,0.0,2.289867,15,91.0,10,97.0,10,97.0,5,5,7,1,1,2,0,17,0,0


In [7]:
X_test = data_test.select_dtypes(exclude=['object']).drop(columns = ['case'])
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)


In [8]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,11.73897,44.96915,polesella,1,7,2023,6.912204e-01
1,11.79794,45.06527,rovigo,1,7,2023,3.737962e-01
2,12.41891,44.92276,porto tolle,1,7,2023,3.129395e-01
3,12.64550,45.53937,jesolo,1,7,2023,2.905398e-01
4,11.72573,45.11874,vescovana,1,7,2023,2.901000e-01
...,...,...,...,...,...,...,...
566,12.11704,46.55724,cortina da ampezzo,1,7,2023,4.589483e-08
567,11.93045,46.43842,rocca pietore,1,7,2023,2.361870e-08
568,11.85565,46.37360,falcade,1,7,2023,2.059427e-08
569,11.88464,46.32739,canale da agordo,1,7,2023,1.421783e-08


In [9]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.2916874171585726, 0.7128345091584661, 0.8345479844283598, 0.9597101609376664, 0.983219513129097, 1.0]


In [10]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,11.73897,44.96915,polesella,1,7,2023,6.912204e-01,1
1,11.79794,45.06527,rovigo,1,7,2023,3.737962e-01,1
2,12.41891,44.92276,porto tolle,1,7,2023,3.129395e-01,1
3,12.64550,45.53937,jesolo,1,7,2023,2.905398e-01,0
4,11.72573,45.11874,vescovana,1,7,2023,2.901000e-01,0
...,...,...,...,...,...,...,...,...
566,12.11704,46.55724,cortina da ampezzo,1,7,2023,4.589483e-08,0
567,11.93045,46.43842,rocca pietore,1,7,2023,2.361870e-08,0
568,11.85565,46.37360,falcade,1,7,2023,2.059427e-08,0
569,11.88464,46.32739,canale da agordo,1,7,2023,1.421783e-08,0


In [11]:
results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}.csv", encoding = enc, index = False)

In [12]:
##TODO Visualisation of results